# Phase 4: Explainability (TreeSHAP) & Prescriptive Counterfactual Engine

## Executive Summary & Objectives
In banking and wealth management, predictive scores alone are not enough for relationship managers and executives:
- **Auditors & Regulators** require transparent justifications for risk classifications (fair lending & adverse action explanation).
- **Relationship Managers** need actionable retention playbooks: *"What specific action can I take to reduce this €120,000 customer's churn probability from 84% to 15%?"*

In this phase, we:
1. **Extract Global Feature Attributions** via **TreeSHAP** to identify systemic drivers of deposit flight.
2. **Compute Local Feature Attributions** for individual high-risk accounts.
3. **Build the Prescriptive Counterfactual Engine** to simulate modifiable levers:
   - **Lever 1: Service Complaint Resolution & Goodwill Perk** (`Complain: 1 -> 0, Satisfaction: +1`).
   - **Lever 2: Digital Channel Activation** (`IsActiveMember: 0 -> 1`).
   - **Lever 3: Product Restructuring** (`NumOfProducts: >=3 -> 2`).
   - **Lever 4: Card Tier Upgrade & Fee Waiver** (`CardType: SILVER -> PLATINUM`).

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

from src.config import ARTIFACTS_DIR, FIGURES_DIR, RAW_DATA_PATH
from src.preprocess import load_and_clean_data, split_features_and_target
from src.explainability import BankChurnExplainer

print("Explainability & Counterfactual environment ready.")

## 1. Load Champion Pipeline & Initialize TreeSHAP Explainer

In [ ]:
raw_pipeline = joblib.load(ARTIFACTS_DIR / "raw_pipeline.pkl")
explainer = BankChurnExplainer(raw_pipeline)

print(f"Explainer initialized with {len(explainer.get_feature_names())} engineered features:")
print(explainer.get_feature_names())

## 2. Global Macro Drivers of Churn (TreeSHAP Visualizations)

In [ ]:
display(Image(filename=str(FIGURES_DIR / "shap_global_importance_bar.png")))
display(Image(filename=str(FIGURES_DIR / "shap_summary_beeswarm.png")))

## 3. Local Customer Explanation (Single Account Deep-Dive)

Let's select a representative high-net-worth at-risk customer:

In [ ]:
sample_customer = pd.DataFrame([{
    "CreditScore": 615,
    "Geography": "Germany",
    "Gender": "Female",
    "Age": 47,
    "Tenure": 3,
    "Balance": 135000.0,
    "NumOfProducts": 3,
    "HasCrCard": 1,
    "IsActiveMember": 0,
    "EstimatedSalary": 82000.0,
    "Complain": 1,
    "SatisfactionScore": 2,
    "CardType": "SILVER",
    "PointEarned": 380,
}])

baseline_risk = raw_pipeline.predict_proba(sample_customer)[:, 1][0]
local_attributions = explainer.explain_customer(sample_customer, top_k=8)

print(f"Customer Balance: €{sample_customer['Balance'].iloc[0]:,.2f}")
print(f"Baseline Predicted Churn Risk: {baseline_risk:.2%}")
print("\nTop Local SHAP Drivers:")
for feat, impact in local_attributions.items():
    direction = "↑ Increases Churn" if impact > 0 else "↓ Decreases Churn"
    print(f" -> {feat:30s}: {impact:+.4f} ({direction})")

## 4. The Prescriptive Counterfactual Engine (Retention Playbook)

Now we run the Prescriptive Counterfactual Engine to simulate targeted business interventions:

In [ ]:
rx = explainer.generate_prescriptive_counterfactuals(sample_customer)

print("=" * 65)
print("PRESCRIPTIVE RETENTION PLAYBOOK & COUNTERFACTUAL SIMULATION")
print("=" * 65)
print(f"Baseline Churn Risk:           {rx['baseline_probability']:.2%}")
print(f"Simulated Post-Action Risk:    {rx['simulated_probability']:.2%}")
print(f"Absolute Risk Reduction:       {rx['absolute_risk_drop']:.2%}")
print(f"Relative Risk Reduction:       {rx['risk_reduction_pct']:.1f}%")
print(f"Potential Balance Retained:    €{rx['potential_deposit_retained']:,.2f}")
print(f"Expected Deposits Saved:       €{rx['expected_deposit_saved']:,.2f}")
print("\nRecommended Actions for Relationship Manager:")
for i, act in enumerate(rx['recommended_interventions'], 1):
    print(f" {i}. {act}")
print("=" * 65)